In [1]:
## File created 19 April 2022

In [10]:
import psycopg2
import os
#from passlib.hash import pbkdf2_sha256
# see here: https://passlib.readthedocs.io/en/stable/lib/passlib.hash.pbkdf2_digest.html
from passlib.hash import django_pbkdf2_sha256 as handler

In [11]:
market='kids'
#market='ladies'

In [12]:
## insert correct credentials!

if market=='kids':
    conn = psycopg2.connect("dbname=kkdj20_test user=sebastian host=localhost password=123456")
elif market=='ladies':
    conn = psycopg2.connect("dbname=kraladies user=sebastian host=localhost password=123456")
else:
    print ("something went wrong. no market specified!")    
cur = conn.cursor()
print (f"Attention: Will insert into {market.upper()}!")

Attention: Will insert into KIDS!


In [13]:
def simple_pwd(n_digits=5, uppercase=False):
    import string
    import secrets
    if uppercase:
        alphabet = string.ascii_letters + string.digits
    else:
        alphabet = string.ascii_lowercase + string.digits
    return ''.join(secrets.choice(alphabet) for i in range(n_digits))

In [14]:
def get_password(username, fname, lname, path='/home/sebastian/Computing/kra20/kassier/pw_files', dryrun=True, verbose=True):
    if not os.path.exists(path):
        sys.exit(1)
    else:
        #print ('ok')
        pass
    _fn = '{}_{}.pwd'.format(fname.lower(),lname.lower())
    _fn_full = os.path.join(path, _fn)
    if os.path.exists(_fn_full):
        with open(_fn_full, 'r') as f:
            _pwd = f.readline().split(',')[1].strip()
        if verbose:
            print ("Password='{}'".format(_pwd))
        return _pwd
    else:
        if verbose:
            print(f"Password file not found for user {username} ({fname} {lname})")
        else:
            return None

In [15]:
def get_or_set_password(username, fname, lname, path='/home/sebastian/Computing/kra20/kassier/pw_files', dryrun=True):
    if not os.path.exists(path):
        sys.exit(1)
    else:
        #print ('ok')
        pass
    _fn = '{}_{}.pwd'.format(fname.lower(),lname.lower())
    _fn_full = os.path.join(path, _fn)
    if not os.path.exists(_fn_full):
        print ('File does not exist. Create file and random password')
        _pwd = simple_pwd(n_digits=5)
        if dryrun:
            print ("Dryrun. I don't write to disk. Password={}".format(_pwd))
        else:
            with open(_fn_full, 'w+') as f:
                f.write("{}, {}".format(username, _pwd))
            print ("File created {}; Password='{}'".format(_fn_full, _pwd))
    else:
        with open(_fn_full, 'r') as f:
            _pwd = f.readline().split(',')[1].strip()
        print ("Password='{}'".format(_pwd))
    return _pwd

In [35]:
def username_in_db(username, fname, lname, verbose=True):
    cur.execute("SELECT id, username, first_name, last_name FROM auth_user WHERE username=%s;",(username, ))
    res = cur.fetchall()
    if verbose:
        print (res)
    if not res:
        return False
    else:
        try:
            assert (res[0][2]==fname)
            assert (res[0][3]==lname)
            return True
        except Exception as e:
            print (f"Error {e}: {res[0][2]}=?{fname} and {res[0][3]}=?{lname}")
            sys.exit(0)

In [28]:
def deactivate_normal_users(verbose=True, dryrun=True):
    cur.execute("SELECT id, username, first_name, last_name, is_active, is_staff, is_superuser FROM auth_user;")
    res = cur.fetchall()
    for r in res:
        if verbose:
            print (r)
            if r[4]:
                print (f"   {r[1]} is_active={r[4]}, is_staff={r[5]}, is_superuser={r[6]}")
            if r[5]:
                print (f"   {r[1]} is staff")
            if r[6]:
                print (f"   {r[1]} is superuser")
        if r[4] and not (r[5] or r[6]):
            if verbose:
                print (f"  {r[1]} is active, but not staff or superuser -> deactivate {r[1]}")
            if not dryrun:
                cur.execute("UPDATE auth_user SET is_active=FALSE WHERE id=%s;",(r[0],))
            else:
                print (f"  dryrun! otherwise would set is_active=FALSE for id={r[0]};")
        elif r[4]:
            if verbose:
                print (f"  {r[1]} is active but staff and/or superuser -> keep as is.")
            

In [42]:
def activate_user(username, verbose=True, dryrun=True):
    cur.execute("SELECT id, first_name, last_name FROM auth_user WHERE username=%s",(username, ))
    res = cur.fetchall()
    if not res:
        print (f"Something went wrong. Failed to find {username=} in auth_user!")
        sys.exit(1)
    else:
        if verbose:
            print (f"  ... {username=} is (id={res[0][0]}) {res[0][1]} {res[0][2]}")
    if not dryrun:
        cur.execute("UPDATE auth_user SET is_active=TRUE WHERE id=%s;",(res[0][0],))
    else:
        print (f"  dryrun. Would now activate user_id={res[0][0]} {res[0][1]} {res[0][2]}")

In [43]:
def add_cashier(fname, lname, role=None, dryrun=True, verbose=True):
    print (40*'*')
    username = '{}'.format(fname.lower())
    if username_in_db(username=username, fname=fname, lname=lname, verbose=verbose):
        print (f'{username} ({fname} {lname}) already in auth_user.')
        activate_user(username=username, verbose=verbose, dryrun=dryrun)
        password_raw = get_password(username, fname, lname, dryrun=dryrun, verbose=True)
        return -1
    print ("Adding {} ({} {}) to auth_user ...".format(username, fname,lname) )
    sql_cmd = "INSERT INTO auth_user (password, last_login, is_superuser, username, first_name, last_name, email, is_staff, is_active, date_joined) VALUES (%s, now(),%s,%s,%s,%s,%s,%s,%s, now())"
    password_raw = get_or_set_password(username, fname, lname, dryrun=dryrun)
    pw_hash = handler.hash(password_raw)
    last_login = ''
    is_superuser = False #  If true, has all rights. Dont!
    is_staff = False # If true, can see summary of all vendors
    if role=='su':
        is_staff = True
        print ('create staff user')
    email = ''
    is_active = True
    date_joined = ''
    if not dryrun:
        cur.execute(sql_cmd, (pw_hash, is_superuser, username, fname, lname, email, is_staff, is_active))
        cur.execute("SELECT id FROM auth_user WHERE first_name=%s AND last_name=%s",(fname, lname))
        res = cur.fetchall()
        if not res:
            print ("Something went wrong. Failed to insert {} {} to auth_user".format(fname,lname))
            sys.exit(1)
        else:
            print ("  ... success. {} {} has auth_user_id={}".format(fname,lname,res[0]))
    else:
        print ("Dryrun. ", password_raw, is_superuser, username, fname, lname, email, is_staff, is_active)

In [33]:
!cat kassierer_20250920.txt

# Kassen-Crew 20 September 2025
# 15. Kra-Kiddy Markt
# Vorname, Nachname, Rolle
# Jana, Wagenmann, 
Sarah, Villinger,
# Monja, Kerzenmacher,
Julie, Friedrich,
Gitte, Braun-Muessle,
# Sonja, Ehret-Knupfer,
# Sandra, Hiss, 
Verena, Goehler, su
Jasmin, Vitt,
Tamara, Tritschler,
# Ute, Schwoerer-Mamier, 
# # Misch, Ganter,
# Desiree, Schweizer,
Carina, Dirr, su
Sarina, Weber, su
Katharina, Schueber, 
Janine, Mueller, 
# # Andrea, Fischer, 
Pina, Chiovetta, su
Tini, Schwoerer, su
Tatjana, Blum, 
# Marlena, Knupfer, 
# Tine, Leber, 
Eugenie, Sorgente, 
Jasmin, Mattmueller, 


In [48]:
## first deactivate all normal users:
deactivate_normal_users(dryrun=True)

(25, 'jasmin', 'Jasmin', 'Vitt', True, False, False)
   jasmin is_active=True, is_staff=False, is_superuser=False
  jasmin is active, but not staff or superuser -> deactivate jasmin
  dryrun! otherwise would set is_active=FALSE for id=25
(33, 'tamara', 'Tamara', 'Tritschler', True, False, False)
   tamara is_active=True, is_staff=False, is_superuser=False
  tamara is active, but not staff or superuser -> deactivate tamara
  dryrun! otherwise would set is_active=FALSE for id=33
(34, 'ute', 'Ute', 'Schwoerer-Mamier', False, False, False)
(27, 'carina', 'Carina', 'Dirr', True, True, False)
   carina is_active=True, is_staff=True, is_superuser=False
   carina is staff
  carina is active but staff and/or superuser -> keep as is.
(1, 'sebastian', '', '', True, True, True)
   sebastian is_active=True, is_staff=True, is_superuser=True
   sebastian is staff
   sebastian is superuser
  sebastian is active but staff and/or superuser -> keep as is.
(37, 'andrea', 'Andrea', 'Fischer', False, False,

In [46]:
dryrun = True # Set to False only after testing!
line_nr = 0
skip_n_lines = 3
with open("kassierer_20250920.txt") as infile:
    for _ in range(skip_n_lines):
        next(infile)
    for line in infile:
        line_nr+=1
        w = line.strip().split(",")
        w = [ww.strip() for ww in w]
        if "#" in w[0]:
            print (40*'*')
            print ('I should skip this line...')
            continue
        if len(w) == 3:
            #print (w)
            add_cashier(fname=w[0], lname=w[1], role=w[2], dryrun=dryrun)

****************************************
I should skip this line...
****************************************
[(19, 'sarah', 'Sarah', 'Villinger')]
sarah (Sarah Villinger) already in auth_user.
  ... username='sarah' is (id=19) Sarah Villinger
  dryrun. Would now activate user_id=19 Sarah Villinger
Password='w63iw'
****************************************
I should skip this line...
****************************************
[(21, 'julie', 'Julie', 'Friedrich')]
julie (Julie Friedrich) already in auth_user.
  ... username='julie' is (id=21) Julie Friedrich
  dryrun. Would now activate user_id=21 Julie Friedrich
Password='mmpw1'
****************************************
[(22, 'gitte', 'Gitte', 'Braun-Muessle')]
gitte (Gitte Braun-Muessle) already in auth_user.
  ... username='gitte' is (id=22) Gitte Braun-Muessle
  dryrun. Would now activate user_id=22 Gitte Braun-Muessle
Password='pt7pt'
****************************************
I should skip this line...
************************************

In [47]:
conn.commit() # Without committing, nothing is stored in DB